In [88]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd


import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation

from recbole.model.general_recommender import NeuMF
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger
from collections import defaultdict
import os
from recbole.quick_start import load_data_and_model



In [89]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/sasrec_ml-1m.pth'
)
model.eval()
device = config['device']

12 May 22:05    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = dataset/ml-1mm
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'NDCG', 'Hit', 'Deep_LT_Coverage', 'GiniIndex', 'AveragePopularity', 'ItemCoverage', 'NDCGTail', 'NDCGHead', 'NDCGMid']
topk = [10]
valid_metric = NDCG@10
valid_metric_

In [90]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict



model.eval()
device = config['device']

# ── 1. LOAD RAW METADATA ────────────────────────────────────────────────────
# read the raw file and inspect first

movies = pd.read_csv(
    'dataset/ml-1mm/ml-1mm.item',
    sep='\t',
    engine='python'
)

# rename to simple names
movies = movies.rename(columns={
    'item_id:token':        'item_id',
    'movie_title:token_seq': 'title',
    'release_year:token':   'year',
    'genre:token_seq':      'genre'
})

# genres are space-separated in this file (not pipe-separated)
# e.g. "Animation Children's Comedy" instead of "Animation|Children|Comedy"
# so split on space when building genre pools


ratings = pd.read_csv(
    'dataset/ml-1mm/ml-1mm.inter',
    sep='\t',
    engine='python'
)


# rename to simple names
ratings = ratings.rename(columns={
    'user_id:token':    'user_id',
    'item_id:token':    'item_id',
    'rating:float':     'rating',
    'timestamp:float':  'timestamp'
})


# map raw item IDs to RecBole internal IDs
item_id_map = dataset.field2token_id['item_id']
movies['internal_id'] = movies['item_id'].astype(str).map(item_id_map)
movies = movies.dropna(subset=['internal_id'])
movies['internal_id'] = movies['internal_id'].astype(int)
movies = movies.sort_values('internal_id').reset_index(drop=True)



n_items = dataset.item_num
n_users = dataset.user_num



# ── 2. DEFINE CONCEPT ITEM POOLS ────────────────────────────────────────────
# split each genre string by space, flatten, and deduplicate
all_unique_genres = sorted(set(
    g.strip()
    for genres in movies['genre'].dropna()
    for g in str(genres).split(' ')
    if g.strip()
))


# genre pools — items belonging to each genre
genre_pools = defaultdict(list)
for _, row in movies.iterrows():
    iid = int(row['internal_id'])
    for g in str(row['genre']).split(' '):   
        g = g.strip()
        if g in all_unique_genres:                 
            genre_pools[g].append(iid )



## defining populairty as continus concept(numerical)

In [91]:
# Per-item interaction counts (same as before)
inter_df = pd.DataFrame({
    'user_id': dataset.inter_feat['user_id'].numpy(),
    'item_id': dataset.inter_feat['item_id'].numpy(),
})

item_counts = inter_df.groupby('item_id')['user_id'].count()
pop_map = {int(iid): int(count) for iid, count in item_counts.items()}

counts_arr = np.array([pop_map.get(i, 0) for i in range(n_items)], dtype=np.float32)

# ── Continuous popularity score per item, normalized to [0, 1] ────────────────
# Use log scaling because raw counts are heavy-tailed (a few items dominate)
log_counts        = np.log1p(counts_arr)
item_popularity   = log_counts / log_counts.max()        # shape: [n_items], in [0, 1]
item_popularity[0] = 0.0                                 # padding token has no popularity

print(f"Popularity score: min={item_popularity.min():.4f}, "
      f"mean={item_popularity.mean():.4f}, max={item_popularity.max():.4f}")

Popularity score: min=0.0000, mean=0.5982, max=1.0000


## Defining popularuty as categorical concpet(nich/mid/popular)

In [92]:

# popularity pool — top 10% most interacted items
inter_df = pd.DataFrame({
    'user_id': dataset.inter_feat['user_id'].numpy(),
    'item_id': dataset.inter_feat['item_id'].numpy(),
})


item_counts = inter_df.groupby('item_id')['user_id'].count()
pop_map = {}
for raw_id, count in item_counts.items():
    #iid = item_id_map.get(str(raw_id))
    iid = int(raw_id)

    if iid is not None:
        pop_map[int(iid)] = count

counts_arr = np.array([pop_map.get(i, 0) for i in range(n_items)])


# popularity pool — top 10% most interacted items
# niche pool      — bottom 10% least interacted items (excluding zero-interaction items)
# mid pool        — everything in between

niche_threshold = np.percentile(counts_arr, 20)
pop_threshold   = np.percentile(counts_arr, 80)

niche_pool      = [i for i in range(n_items) if 0 < counts_arr[i] <= niche_threshold]
mid_pool        = [i for i in range(n_items) if niche_threshold < counts_arr[i] < pop_threshold]
popularity_pool = [i for i in range(n_items) if counts_arr[i] >= pop_threshold]

print(f"Popularity pool: {len(popularity_pool)} items (≥ {pop_threshold:.0f} interactions)")
print(f"Mid pool:        {len(mid_pool)} items")
print(f"Niche pool:      {len(niche_pool)} items (1–{niche_threshold:.0f} interactions)")


Popularity pool: 686 items (≥ 459 interactions)
Mid pool:        2042 items
Niche pool:      688 items (1–35 interactions)


In [93]:
# era pools
movies['release_year'] = pd.to_numeric(movies['release_year'], errors='coerce')

classic_pool   = [int(r['internal_id']) for _, r in movies.iterrows() if r['release_year'] < 1970]
retro_pool     = [int(r['internal_id']) for _, r in movies.iterrows() if 1970 <= r['release_year'] < 1990]
modern_pool    = [int(r['internal_id']) for _, r in movies.iterrows() if 1990 <= r['release_year'] < 2000]
contemporary_pool = [int(r['internal_id']) for _, r in movies.iterrows() if r['release_year'] >= 2000]


era_pools = {
    'classic': classic_pool,
    'retro': retro_pool,
    'modern': modern_pool,
    'contemporary': contemporary_pool
}


In [94]:

# Concepts we'll predict (in fixed order)
genre_concepts = list(all_unique_genres)              # e.g. ['Action', 'Comedy', ...]
scalar_concepts = ['popularity','mid','niche',
                   'classic', 'retro', 'modern', 'contemporary']
concept_names = genre_concepts + scalar_concepts
N_CONCEPTS = len(concept_names)

print(f"Total concepts: {N_CONCEPTS}")

# Lookup: item_id -> set of genres
item_to_genres = {}
for _, row in movies.iterrows():
    item_to_genres[int(row['internal_id'])] = set(row['genre'].split('|'))




# Lookup: item_id -> year bucket
item_to_era = {}
for _, row in movies.iterrows():
    y = row['release_year']
    if pd.isna(y):                  era = None
    elif y < 1970:                  era = 'classic'
    elif y < 1990:                  era = 'retro'
    elif y < 2000:                  era = 'modern'
    else:                           era = 'contemporary'
    item_to_era[int(row['internal_id'])] = era



# Sets for popularity / niche
pop_set   = set(popularity_pool)
niche_set = set(niche_pool)
mid_set   = set(mid_pool)


Total concepts: 25


In [95]:



def compute_user_concepts(item_seq):
    """item_seq: list/array of item IDs (padding 0s allowed). Returns [N_CONCEPTS] vector."""
    items = [i for i in item_seq if i != 0]
    if len(items) == 0:
        return np.zeros(N_CONCEPTS, dtype=np.float32)

    L = len(items)
    vec = np.zeros(N_CONCEPTS, dtype=np.float32)

    # genre fractions
    genre_idx = {g: i for i, g in enumerate(genre_concepts)}
    for it in items:
        for g in item_to_genres.get(it, []):
            for g_s in g.split(' '):
                if g_s in genre_idx:
                    vec[genre_idx[g_s]] += 1.0
                    #break
    vec[:len(genre_concepts)] /= L     # normalize to fractions

    # popularity / Mid/ niche fractions

    # ── Popularity: average popularity score of items in history ─────────────
    pop_offset       = len(genre_concepts)
    #vec[pop_offset]  = float(np.mean([item_popularity[it] for it in items]))
    vec[len(genre_concepts) + 0] = sum(1 for it in items if it in pop_set)   / L
    vec[len(genre_concepts) + 1] = sum(1 for it in items if it in mid_set) / L
    vec[len(genre_concepts) + 2] = sum(1 for it in items if it in niche_set) / L

    # era fractions
    era_offset = len(genre_concepts) + 3
    era_idx = {'classic': 0, 'retro': 1, 'modern': 2, 'contemporary': 3}

    era_count = 0
    for it in items:
        e = item_to_era.get(it)
        if e in era_idx:
            vec[era_offset + era_idx[e]] += 1.0
            era_count += 1
    if era_count > 0:
        vec[era_offset:era_offset+4] /= era_count

    return vec
    

In [45]:
movies

,item_id,movie_title,release_year,genre,internal_id
0,1193,One Flew Over the Cuckoo's Nest,1975,Drama,1
1,661,James and the Giant Peach,1996,Animation Children's Musical,2
2,914,My Fair Lady,1964,Musical Romance,3
3,3408,Erin Brockovich,2000,Drama,4
4,2355,"Bug's Life, A",1998,Animation Children's Comedy,5
...,...,...,...,...,...
3411,2833,Lucie Aubrac,1997,Romance War,3412
3412,3207,"Snows of Kilimanjaro, The",1952,Adventure,3413
3413,3533,"Actor's Revenge, An (Yukinojo Henge)",1963,Drama,3414
3414,2777,Cobra,1925,Drama,3415


In [24]:
for batch in train_data:
    batch
    break

In [66]:
input_=torch.tensor([1,2,3,4])

In [69]:
compute_user_concepts(input_.tolist())

array([0.  , 0.  , 0.25, 0.25, 0.  , 0.  , 0.  , 0.5 , 0.  , 0.  , 0.  ,
       0.5 , 0.  , 0.25, 0.  , 0.  , 0.  , 0.  , 0.5 , 0.5 , 0.  , 0.25,
       0.25, 0.25, 0.25], dtype=float32)

In [49]:
from recbole.utils import build_lookups, build_cache, seq_to_concepts
import pickle
L=build_lookups(dataset, 'ml-1mm')
item_concepts,_=build_cache('ml-1mm', dataset)

  pop_set:   342 items (≥ 759 interactions)
  mid_set:   2716 items
  niche_set: 358 items (1–18 interactions)
  Items in metadata file: 3883, matched to internal IDs: 3416


  N_CONCEPTS = 25  (18 genres + 7 scalars)
  genres: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
[build] Building concept lookups...
  pop_set:   342 items (≥ 759 interactions)
  mid_set:   2716 items
  niche_set: 358 items (1–18 interactions)
  Items in metadata file: 3883, matched to internal IDs: 3416
  N_CONCEPTS = 25  (18 genres + 7 scalars)
  genres: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
[build] Computing per-item concept vectors for 3417 items...

[build] Saved per-item concept matrix ((3417, 25)) → ./dataset/ml-1mm/saved_concept_individual_items.pkl

[build] Diagnostics — items with NO ...
    ... genre features:  1 (should be ~1, padding)
    ... era features:   

In [96]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SASRecCBM(nn.Module):
    """
    Predictive concept bottleneck on top of frozen SASRec.

    Pipeline:  h → concept predictor → ĉ → reconstructor → ĥ → item scores
    """
    def __init__(self, sasrec, n_concepts, hidden_size=128):
        super().__init__()
        self.sasrec = sasrec                         # frozen encoder
        for p in self.sasrec.parameters():
            p.requires_grad = False

        self.hidden_size = hidden_size
        self.n_concepts  = n_concepts

        # h -> ĉ
        self.concept_predictor = nn.Sequential(
        nn.Linear(hidden_size, 256),
        #nn.BatchNorm1d(256),
        nn.LayerNorm(256), 
        #nn.LeakyReLU(0.1),
        nn.GELU(),                         # ← matches SASRec's activation
        nn.Dropout(0.1),

        nn.Linear(256, 128),
        #nn.BatchNorm1d(128),
        nn.LayerNorm(128), 
        #nn.LeakyReLU(0.1),
        nn.GELU(),                         # ← matches SASRec's activation

        nn.Dropout(0.1),

        nn.Linear(128, n_concepts),
        nn.Sigmoid(),
    )

        self.reconstructor = nn.Sequential(
        #nn.Linear(n_concepts, 64),
        nn.Linear(n_concepts, 128),

        #nn.BatchNorm1d(64),
        nn.LayerNorm(128),                 # ← LayerNorm instead of BatchNorm

        #nn.LeakyReLU(0.1),
        nn.GELU(),                         # ← matches SASRec's activation

        nn.Linear(128, hidden_size),
)
       

       

    def encode(self, item_seq, item_seq_len):
        with torch.no_grad():
            return self.sasrec.forward(item_seq, item_seq_len)   # [B, 128]

    def forward(self, item_seq, item_seq_len):
        h     = self.encode(item_seq, item_seq_len)              # [B, 128]
        c_hat = self.concept_predictor(h)                        # [B, N_CONCEPTS]
        h_hat = self.reconstructor(c_hat)                        # [B, 128]
        return h, c_hat, h_hat

    def score_items(self, h_hat):
        # use SASRec's own item embedding table as the prediction head
        item_emb = self.sasrec.item_embedding.weight   
        #torch.matmul(h_hat, item_emb.transpose(0, 1))  # [B n_items]          # [n_items, 128]
        return torch.matmul(h_hat, item_emb.transpose(0, 1))                               # [B, n_items]

# SASRec-only eval function

In [97]:
@torch.no_grad()
def evaluate_sasrec(sasrec, eval_data, k_values=(5, 10, 20)):
    """
    Evaluate frozen SASRec directly (no bottleneck).
    Uses the same masking/ranking logic as evaluate_cbm for a fair comparison.
    """
    sasrec.eval()

    hits  = {k: 0   for k in k_values}
    ndcgs = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n_users = 0

    for batch in eval_data:
        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch
        interaction  = interaction.to(device)

        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        # Forward through SASRec directly, score against item embeddings
        h        = sasrec.forward(item_seq, item_seq_len)  
                  # [B, 128]
        item_emb = sasrec.item_embedding.weight                      # [n_items, 128]
        #logits   = h @ item_emb.T   

        logits=sasrec.full_sort_predict(interaction)                        

        # Same masking as evaluate_cbm
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        for k in k_values:
            in_top_k  = (rank <= k)
            hits[k]  += in_top_k.sum().item()
            ndcgs[k] += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()

        mrr_sum += (1.0 / rank.float()).sum().item()
        n_users += target_item.size(0)

    return {
        'hr':   {k: hits[k]  / n_users for k in k_values},
        'ndcg': {k: ndcgs[k] / n_users for k in k_values},
        'mrr':  mrr_sum / n_users,
    }

## TMP evalaution metrics 

In [ ]:
'''
@torch.no_grad()
def evaluate_sasrec(sasrec, eval_data, k_values=(5, 10, 20)):
    """
    Evaluate frozen SASRec directly (no bottleneck).
    Uses the same masking/ranking logic as evaluate_cbm for a fair comparison.
    """
    sasrec.eval()

    hits  = {k: 0   for k in k_values}
    ndcgs = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n_users = 0

    for batch in eval_data:
        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch
        interaction  = interaction.to(device)

        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        # Forward through SASRec directly, score against item embeddings
        # 1. Get the sequence of hidden states
        seq_output = sasrec.forward(item_seq, item_seq_len) 
        
        # 2. Extract ONLY the last item's hidden state [B, H]
        # We use item_seq_len - 1 to get the actual last interaction index
        h = seq_output[torch.arange(item_seq.size(0)), item_seq_len - 1]

        # 3. Calculate raw scores
        item_emb = sasrec.item_embedding.weight
        logits = h @ item_emb.T 

        # 4. Get the score of the actual target item FIRST
        target_scores = logits.gather(1, target_item.view(-1, 1))

        # 5. Now mask history and padding in the logits
        scores = logits.clone()
        scores[:, 0] = -float('inf') # Mask padding
        scores.scatter_(1, item_seq, -float('inf')) # Mask history
        rank          = (scores > target_scores).sum(dim=1) + 1

        for k in k_values:
            in_top_k  = (rank <= k)
            hits[k]  += in_top_k.sum().item()
            ndcgs[k] += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()

        mrr_sum += (1.0 / rank.float()).sum().item()
        n_users += target_item.size(0)

    return {
        'hr':   {k: hits[k]  / n_users for k in k_values},
        'ndcg': {k: ndcgs[k] / n_users for k in k_values},
        'mrr':  mrr_sum / n_users,
    }
'''

## Training and Evalaution for concept predictor

In [103]:
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import pearsonr

device = next(model.parameters()).device

cbm = SASRecCBM(model, n_concepts=N_CONCEPTS, hidden_size=model.hidden_size).to(device)
opt = torch.optim.Adam(
    [p for p in cbm.parameters() if p.requires_grad], lr=1e-3
)

LAMBDA_CONCEPT = 1

LAMBDA_RECON   = 4

LAMBDA_ACC= 1

N_EPOCHS       = 300
K_VALUES       = [ 10]

TOP_K_CONCEPTS = 3


# ── EVAL FUNCTION ─────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_cbm(cbm, eval_data, k_values=(5, 10, 20), top_k_concepts=3):
    cbm.eval()

    total_rec, total_con, n_batches = 0., 0., 0
    hits  = {k: 0   for k in k_values}
    ndcgs = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n_users = 0

    all_pred, all_true = [], []
    topk_true_all, topk_pred_all, topk_idx_all = [], [], []
    topk_exact_correct   = 0.0
    topk_partial_correct = 0.0

    for batch in eval_data:
        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch
        interaction  = interaction.to(device)

        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        gt_concepts = torch.stack([
            torch.from_numpy(compute_user_concepts(seq.cpu().tolist()))
            for seq in item_seq
        ]).to(device)

        _, c_hat, h_hat = cbm(item_seq, item_seq_len)
        logits          = cbm.score_items(h_hat)

        loss_rec = F.cross_entropy(logits, target_item)
        loss_con = F.binary_cross_entropy(c_hat, gt_concepts)
        total_rec += loss_rec.item()
        total_con += loss_con.item()
        n_batches += 1

        # ── top-k concept value capture ───────────────────────────────────────
        true_topk_vals, true_topk_idx = gt_concepts.topk(top_k_concepts, dim=1)
        pred_on_topk                  = c_hat.gather(1, true_topk_idx)

        topk_true_all.append(true_topk_vals.cpu().numpy())
        topk_pred_all.append(pred_on_topk.cpu().numpy())
        topk_idx_all.append(true_topk_idx.cpu().numpy())


        # ── strict and partial top-k set-match accuracy ──────────────────────
        true_sorted = true_topk_idx.sort(dim=1).values
        pred_sorted = c_hat.topk(top_k_concepts, dim=1).indices.sort(dim=1).values

        exact_match = (pred_sorted == true_sorted).all(dim=1).float()
        topk_exact_correct += exact_match.sum().item()

        for u in range(true_sorted.size(0)):
            ov = len(set(pred_sorted[u].tolist()) & set(true_sorted[u].tolist()))
            topk_partial_correct += ov / top_k_concepts

        all_pred.append(c_hat.cpu().numpy())
        all_true.append(gt_concepts.cpu().numpy())

        # ── recommendation metrics ────────────────────────────────────────────
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        for k in k_values:
            in_top_k  = (rank <= k)
            hits[k]  += in_top_k.sum().item()
            ndcgs[k] += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        mrr_sum += (1.0 / rank.float()).sum().item()
        n_users += target_item.size(0)

    # ── aggregate ─────────────────────────────────────────────────────────────
    pred = np.vstack(all_pred)
    true = np.vstack(all_true)
    topk_true = np.vstack(topk_true_all)
    topk_pred = np.vstack(topk_pred_all)
    topk_idx  = np.vstack(topk_idx_all)

    overall_mae   = float(np.mean(np.abs(pred - true)))
    baseline_mae  = float(np.mean(np.abs(true - true.mean(axis=0, keepdims=True))))
    topk_mae      = float(np.mean(np.abs(topk_pred - topk_true)))
    topk_recovery = float(np.mean(
        np.clip(topk_pred / np.clip(topk_true, 1e-6, None), 0, 2)
    ))
    topk_corr     = float(pearsonr(topk_true.flatten(), topk_pred.flatten()).statistic)

    return {
        'rec_loss':         total_rec / n_batches,
        'con_loss':         total_con / n_batches,
        'concept_mae':      overall_mae,
        'baseline_mae':     baseline_mae,
        'topk_mae':         topk_mae,
        'topk_recovery':    topk_recovery,
        'topk_corr':        topk_corr,
        'topk_exact_acc':   topk_exact_correct   / n_users,
        'topk_partial_acc': topk_partial_correct / n_users,
        'topk_true':        topk_true,
        'topk_pred':        topk_pred,
        'topk_idx':         topk_idx,
        'hr':               {k: hits[k]  / n_users for k in k_values},
        'ndcg':             {k: ndcgs[k] / n_users for k in k_values},
        'mrr':              mrr_sum / n_users,
    }
## Evaluate the frozen SASRec as a baseline before training the CBM

print("Evaluating frozen SASRec baseline...")
sasrec_baseline = evaluate_sasrec(model, test_data, k_values=K_VALUES)
print(f"  SASRec  HR@10={sasrec_baseline['hr'][10]:.4f}  "
      f"NDCG@10={sasrec_baseline['ndcg'][10]:.4f}  "
      f"MRR={sasrec_baseline['mrr']:.4f}\n")


BEST_METRIC = 'hr'   ## metrics name
BEST_K      = 10     # for hr/ndcg
SAVE_PATH   = 'best_cbm_ml-1m_SASREC_3pop_TMP.pt'

best_score = -float('inf')   # use float('inf') if tracking a loss/MAE (lower is better)


# ── TRAINING LOOP ─────────────────────────────────────────────────────────────
history = []

for epoch in range(N_EPOCHS):
    cbm.train()
    total_rec, total_con, n_batches = 0., 0., 0

    for batch in train_data:
        batch        = batch.to(device)
        item_seq     = batch['item_id_list']
        item_seq_len = batch['item_length']
        target_item  = batch['item_id']
        
        #print(f'item_seq.shape: {item_seq.shape}')
        gt_concepts = torch.stack([
            torch.from_numpy(compute_user_concepts(seq.cpu().tolist()))
            for seq in item_seq
        ]).to(device)
       
        h, c_hat, h_hat = cbm(item_seq, item_seq_len)
        logits          = cbm.score_items(h_hat)

        loss_rec = F.cross_entropy(logits, target_item)

        loss_recon = F.mse_loss(h_hat, h.detach())


        loss_con = F.binary_cross_entropy(c_hat, gt_concepts)
        #loss     = loss_rec + LAMBDA_CONCEPT * loss_con

        loss =  LAMBDA_CONCEPT * loss_con + LAMBDA_RECON * loss_recon


        opt.zero_grad()
        loss.backward()
        opt.step()

        total_rec += loss_rec.item()
        total_con += loss_con.item()
        n_batches += 1

    train_rec = total_rec / n_batches
    train_con = total_con / n_batches

    # Evaluate on test set after this epoch
    test = evaluate_cbm(cbm, test_data,
                        k_values=K_VALUES, top_k_concepts=TOP_K_CONCEPTS)
    
    # Inside the loop, after `test = evaluate_cbm(...)`:
    current_score = test['hr'][BEST_K]   # or test['ndcg'][BEST_K], test['mrr'], etc.

    if current_score > best_score:
        best_score = current_score
        torch.save({
        'epoch': epoch + 1,
        'model_state_dict': cbm.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'metrics': test,
        'n_concepts': N_CONCEPTS,
        'hidden_size': model.hidden_size,
        }, SAVE_PATH)
        print(f"  ↳ Saved best model (HR@{BEST_K}={current_score:.4f})")
    history.append({'epoch':     epoch + 1,
                    'train_rec': train_rec,
                    'train_con': train_con,
                    'rec_loss':         test['rec_loss'],
                    'con_loss':         test['con_loss'],
                    'concept_mae':      test['concept_mae'],
                    'baseline_mae':     test['baseline_mae'],
                    'topk_mae':         test['topk_mae'],
                    'topk_recovery':    test['topk_recovery'],
                    'topk_corr':        test['topk_corr'],
                    'hr':               test['hr'],
                    'ndcg':             test['ndcg'],
                    'mrr':              test['mrr']})

    print(
    f"Epoch {epoch+1:2d} | "
    f"train: rec={train_rec:.4f} con={train_con:.4f} | "
    f"test: rec={test['rec_loss']:.4f} con={test['con_loss']:.4f} | "
    f"MAE={test['concept_mae']:.4f} (base={test['baseline_mae']:.4f}) | "
    f"top{TOP_K_CONCEPTS}_acc={test['topk_exact_acc']:.3f} "
    f"(partial={test['topk_partial_acc']:.3f}) | "
    
    f"HR@10={test['hr'][10]:.4f} NDCG@10={test['ndcg'][10]:.4f}"
)




Evaluating frozen SASRec baseline...


  SASRec  HR@10=0.2969  NDCG@10=0.1714  MRR=0.1480

  ↳ Saved best model (HR@10=0.2349)
Epoch  1 | train: rec=6.0825 con=0.2918 | test: rec=6.1624 con=0.2764 | MAE=0.0795 (base=0.0675) | top3_acc=0.249 (partial=0.711) | HR@10=0.2349 NDCG@10=0.1244
  ↳ Saved best model (HR@10=0.2596)
Epoch  2 | train: rec=5.6247 con=0.2702 | test: rec=6.0992 con=0.2732 | MAE=0.0752 (base=0.0675) | top3_acc=0.236 (partial=0.704) | HR@10=0.2596 NDCG@10=0.1429
  ↳ Saved best model (HR@10=0.2666)
Epoch  3 | train: rec=5.5714 con=0.2686 | test: rec=6.0756 con=0.2710 | MAE=0.0724 (base=0.0675) | top3_acc=0.235 (partial=0.704) | HR@10=0.2666 NDCG@10=0.1464
  ↳ Saved best model (HR@10=0.2727)
Epoch  4 | train: rec=5.5504 con=0.2675 | test: rec=6.0711 con=0.2703 | MAE=0.0717 (base=0.0675) | top3_acc=0.237 (partial=0.705) | HR@10=0.2727 NDCG@10=0.1503
  ↳ Saved best model (HR@10=0.2786)
Epoch  5 | train: rec=5.5360 con=0.2665 | test: rec=6.0568 con=0.2696 | MAE=0.0710 (base=0.0675) | top3_acc=0.239 (partial=0.706

KeyboardInterrupt: 

In [ ]:
pd.DataFrame(history).to_csv('cbm_training_history_SASREC_ml-1m.csv', index=False)




## Loading CBM model

In [104]:
# Rebuild the architecture first (must match what you trained)
cbm = SASRecCBM(model, n_concepts=N_CONCEPTS, hidden_size=model.hidden_size).to(device)


SAVE_PATH   = 'best_cbm_ml-1m_SASREC_3pop_TMP.pt'
# Load the checkpoint
checkpoint = torch.load(SAVE_PATH, map_location=device)
cbm.load_state_dict(checkpoint['model_state_dict'])
cbm.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best metrics: HR@10={checkpoint['metrics']['hr'][10]:.4f}")

/tmp/ipykernel_645238/2556168713.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(SAVE_PATH, map_location=device)


Loaded model from epoch 7
Best metrics: HR@10=0.2856


In [105]:
# ── ERA EXPOSURE HELPER ───────────────────────────────────────────────────────
def era_exposure(recs, era_pools):
    """
    For each era, fraction of recommended items that fall in that era's pool.
    `recs` shape: [n_users, k]
    Returns: dict {era_name: rate}
    """
    rates = {}
    for era, pool in era_pools.items():
        pool_arr   = np.array(pool)
        rates[era] = float(np.isin(recs, pool_arr).mean())
    return rates

## Steering

In [108]:
import numpy as np
import torch
import torch.nn.functional as F


# ── CONFIGURATION ─────────────────────────────────────────────────────────────
POP_CONCEPT_IDX = concept_names.index('mid')   # index of the concept to steer (popularity in this case)
SCALE_FACTOR    = 1.03
K_FOR_TOPK      = 10
K_FOR_METRICS   = 10        # the K used for HR@K and NDCG@K


# ── STEERING-AWARE EVALUATOR (rec metrics + popularity exposure) ──────────────
@torch.no_grad()
def evaluate_steered(cbm, eval_data,
                     k_metric=20, k_recs=20,
                     concept_idx=None, scale=1.0):
    """
    Single pass over eval_data that returns:
      - HR@k_metric
      - NDCG@k_metric
      - popularity_exposure   (over top-k_recs items)
      - recs                  ([n_users, k_recs] item ids, useful for coverage/Gini)
    """
    cbm.eval()

    hits     = 0
    ndcg_sum = 0.0
    n_users  = 0

    #pop_pool_arr = np.array(list(popularity_pool))
    pop_count    = 0
    total_recs   = 0

    all_recs = []

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        h, c_hat, _ = cbm(item_seq, item_seq_len)

        # ── Apply steering ────────────────────────────────────────────────────
        if concept_idx is not None:
            c_hat = c_hat.clone()
            c_hat[:, concept_idx] = c_hat[:, concept_idx] * scale

        h_hat  = cbm.reconstructor(c_hat)
        logits = cbm.score_items(h_hat)

        # Mask padding and user history
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        # ── HR@k_metric and NDCG@k_metric ─────────────────────────────────────
        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        in_top_k  = (rank <= k_metric)
        hits     += in_top_k.sum().item()
        ndcg_sum += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        n_users  += target_item.size(0)

        # ── Top-k_recs items (for popularity exposure / coverage / Gini) ──────
        topk_items = scores.topk(k_recs, dim=1).indices.cpu().numpy()
        all_recs.append(topk_items)
        #pop_count  += np.isin(topk_items, pop_pool_arr).sum()
        total_recs += topk_items.size

    return {
        f'hr@{k_metric}':   hits / n_users,
        f'ndcg@{k_metric}': ndcg_sum / n_users,
        #'pop_exposure':     pop_count / total_recs,
        'recs':             np.concatenate(all_recs, axis=0),
    }


# ── COVERAGE + GINI (unchanged) ───────────────────────────────────────────────
def coverage(recs, n_items):
    unique_recommended = np.unique(recs)
    unique_recommended = unique_recommended[unique_recommended != 0]
    return len(unique_recommended) / (n_items - 1)


def gini(recs, n_items):
    counts = np.bincount(recs.flatten(), minlength=n_items).astype(np.float64)
    counts = counts[1:]
    counts = np.sort(counts)
    n      = len(counts)
    if counts.sum() == 0:
        return 0.0
    cum = np.cumsum(counts)
    return (2 * np.sum((np.arange(1, n + 1)) * counts) - (n + 1) * cum[-1]) \
           / (n * cum[-1])
def avg_popularity(recs, item_popularity):
    """Mean popularity score of recommended items.
    recs: [n_users, k] item IDs
    item_popularity: [n_items] popularity scores
    """
    flat = recs.flatten()
    flat = flat[flat != 0]   # drop padding
    return float(item_popularity[flat].mean())

def tier_exposure_fast(recs, popularity_pool, niche_pool, n_items):
    """Vectorized version — much faster for large recs matrices."""
    pop_mask   = np.zeros(n_items, dtype=bool); pop_mask[list(popularity_pool)]   = True
    niche_mask = np.zeros(n_items, dtype=bool); niche_mask[list(niche_pool)]      = True

    flat = recs.flatten()
    flat = flat[flat != 0]

    is_pop   = pop_mask[flat]
    is_niche = niche_mask[flat]
    is_mid   = ~(is_pop | is_niche)

    return {
        'pop_rate':   float(is_pop.mean()),
        'mid_rate':   float(is_mid.mean()),
        'niche_rate': float(is_niche.mean()),
    }
# ── RUN THE EXPERIMENT ────────────────────────────────────────────────────────
n_items = cbm.n_items if hasattr(cbm, 'n_items') else model.n_items

print("Evaluating baseline (no steering)...")
base = evaluate_steered(cbm, test_data,
                        k_metric=K_FOR_METRICS, k_recs=K_FOR_TOPK)

print(f"Evaluating steered (popularity × {SCALE_FACTOR})...")
steer = evaluate_steered(cbm, test_data,
                         k_metric=K_FOR_METRICS, k_recs=K_FOR_TOPK,
                         concept_idx=POP_CONCEPT_IDX, scale=SCALE_FACTOR)

# Coverage + Gini from the rec lists
cov_base,  gini_base  = coverage(base ['recs'], n_items), gini(base ['recs'], n_items)
cov_steer, gini_steer = coverage(steer['recs'], n_items), gini(steer['recs'], n_items)



# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print("\n── RESULTS ─────────────────────────────────")
print(f"{'Metric':<18} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 52)

for key in [f'hr@{K_FOR_METRICS}', f'ndcg@{K_FOR_METRICS}']:
    b, s = base[key], steer[key]
    print(f"{key:<18} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")



print(f"{'coverage':<18} {cov_base :>10.4f} {cov_steer:>10.4f} {cov_steer-cov_base :>+10.4f}")
print(f"{'gini':<18} {gini_base:>10.4f} {gini_steer:>10.4f} {gini_steer-gini_base:>+10.4f}")





# After computing cov_base / gini_base / cov_steer / gini_steer
avgpop_base  = avg_popularity(base['recs'],  item_popularity)
avgpop_steer = avg_popularity(steer['recs'], item_popularity)

# Add to the print block
print(f"{'avg_popularity':<18} {avgpop_base:>10.4f} {avgpop_steer:>10.4f} "
      f"{avgpop_steer-avgpop_base:>+10.4f}")




tiers_base  = tier_exposure_fast(base ['recs'], popularity_pool, niche_pool, n_items)
tiers_steer = tier_exposure_fast(steer['recs'], popularity_pool, niche_pool, n_items)

print("\n── POPULARITY TIER EXPOSURE ────────────────")
print(f"{'Tier':<18} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 52)
for tier in ['pop_rate', 'mid_rate', 'niche_rate']:
    b, s = tiers_base[tier], tiers_steer[tier]
    print(f"{tier:<18} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")


Evaluating baseline (no steering)...
Evaluating steered (popularity × 1.03)...

── RESULTS ─────────────────────────────────
Metric               Baseline    Steered          Δ
────────────────────────────────────────────────────
hr@10                  0.2856     0.2838    -0.0018
ndcg@10                0.1578     0.1559    -0.0019
coverage               0.6256     0.6393    +0.0138
gini                   0.7542     0.7443    -0.0099
avg_popularity         0.7974     0.7930    -0.0044

── POPULARITY TIER EXPOSURE ────────────────
Tier                 Baseline    Steered          Δ
────────────────────────────────────────────────────
pop_rate               0.7004     0.6838    -0.0166
mid_rate               0.2980     0.3145    +0.0164
niche_rate             0.0016     0.0018    +0.0002


## Era concept

In [ ]:
concept_names.index('classic')

In [109]:
@torch.no_grad()
def evaluate_era_steered(cbm, eval_data, era_pools, era_concept_idx,
                         target_era, boost=2.0, suppress=0.3,
                         k_metric=20, k_recs=20):
    """
    Steer recommendations toward a specific era by boosting that era's concept
    activation and suppressing the others.
    
    target_era:  one of the keys in era_pools / era_concept_idx
    boost:       multiplier for the target era's concept (>1 = amplify)
    suppress:    multiplier for non-target era concepts (<1 = suppress)
    """
    cbm.eval()
    target_idx = era_concept_idx[target_era]

    hits = 0; ndcg_sum = 0.0; n_users = 0
    all_recs = []

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        h, c_hat, _ = cbm(item_seq, item_seq_len)

        # ── ERA STEERING ──────────────────────────────────────────────────────
        c_hat = c_hat.clone()
        c_hat[:, target_idx] = c_hat[:, target_idx] * boost

        

        h_hat  = cbm.reconstructor(c_hat)
        logits = cbm.score_items(h_hat)

        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        # HR / NDCG
        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1
        in_top_k      = (rank <= k_metric)
        hits         += in_top_k.sum().item()
        ndcg_sum     += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        n_users      += target_item.size(0)

        # Recs
        topk_items = scores.topk(k_recs, dim=1).indices.cpu().numpy()
        all_recs.append(topk_items)

    recs = np.concatenate(all_recs, axis=0)
    

    # Era exposure
    era_rates = {era: float(np.isin(recs, np.array(pool)).mean())
                 for era, pool in era_pools.items()}

    return {
        f'hr@{k_metric}':   hits / n_users,
        f'ndcg@{k_metric}': ndcg_sum / n_users,
        'era_exposure':     era_rates,
        'recs':             recs,
    }


# ── RUN ───────────────────────────────────────────────────────────────────────
ERA_CONCEPT_IDX = {        # <-- fill in your actual concept indices
    'classic':       21,
    'retro':         22,
    'modern':        23,
    'contemporary':  24,
}
TARGET_ERA = 'modern'     # which era to steer toward
BOOST      = 0.75          # amplify target era concept
SUPPRESS   = 1           # suppress other era concepts
K_METRIC   = 10
K_RECS     = 10

# Baseline (no steering) — re-use the function with boost=1, suppress=1
print("Evaluating baseline (no steering)...")
base = evaluate_era_steered(cbm, test_data, era_pools, ERA_CONCEPT_IDX,
                            target_era=TARGET_ERA, boost=1.0, suppress=1.0,
                            k_metric=K_METRIC, k_recs=K_RECS)

print(f"Evaluating steered toward '{TARGET_ERA}' (boost×{BOOST}, suppress×{SUPPRESS})...")
steer = evaluate_era_steered(cbm, test_data, era_pools, ERA_CONCEPT_IDX,
                             target_era=TARGET_ERA, boost=BOOST, suppress=SUPPRESS,
                             k_metric=K_METRIC, k_recs=K_RECS)


# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print("\n── ACCURACY ────────────────────────────────")
print(f"{'Metric':<14} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 48)
for key in [f'hr@{K_METRIC}', f'ndcg@{K_METRIC}']:
    b, s = base[key], steer[key]
    print(f"{key:<14} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")

print("\n── ERA EXPOSURE ────────────────────────────")
print(f"{'Era':<14} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 48)
for era in era_pools.keys():
    b, s = base['era_exposure'][era], steer['era_exposure'][era]
    marker = '  ←' if era == TARGET_ERA else ''
    print(f"{era:<14} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}{marker}")

Evaluating baseline (no steering)...
Evaluating steered toward 'modern' (boost×0.75, suppress×1)...

── ACCURACY ────────────────────────────────
Metric           Baseline    Steered          Δ
────────────────────────────────────────────────
hr@10              0.2856     0.2156    -0.0700
ndcg@10            0.1578     0.1173    -0.0405

── ERA EXPOSURE ────────────────────────────
Era              Baseline    Steered          Δ
────────────────────────────────────────────────
classic            0.0916     0.1980    +0.1064
retro              0.2840     0.2602    -0.0238
modern             0.5038     0.4180    -0.0858  ←
contemporary       0.1206     0.1239    +0.0032


In [ ]:
@torch.no_grad()
def evaluate_concept_steered(cbm, eval_data, concept_pools, concept_idx_map,
                             target_concepts, boost=2.0, suppress=1.0,
                             k_metric=20, k_recs=20):
    """
    Steer recommendations toward one or more concepts.
    
    target_concepts:    list of concept names to amplify (e.g. ['Horror'] or
                        ['classic'] or ['Action', 'Thriller'])
    concept_pools:      dict mapping concept name -> set of item IDs that match
                        that concept. Used only for exposure measurement, not steering.
    concept_idx_map:    dict mapping concept name -> column index in c_hat
    boost:              multiplier for target concept activations (>1 = amplify)
    suppress:           multiplier for non-target concept activations (<1 = suppress, 1.0 = leave alone)
    """
    cbm.eval()
    target_idxs = [concept_idx_map[c] for c in target_concepts]
    target_idxs_t = torch.tensor(target_idxs, device=device)

    hits = 0; ndcg_sum = 0.0; n_users = 0
    all_recs = []

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        h, c_hat, _ = cbm(item_seq, item_seq_len)

        # ── STEERING ──────────────────────────────────────────────────────────
        c_hat = c_hat.clone()
        if suppress != 1.0:
            c_hat = c_hat * suppress              # suppress everything first
        c_hat[:, target_idxs_t] = c_hat[:, target_idxs_t] * (boost / suppress) \
                                  if suppress != 1.0 else c_hat[:, target_idxs_t] * boost

        h_hat  = cbm.reconstructor(c_hat)
        logits = cbm.score_items(h_hat)

        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1
        in_top_k      = (rank <= k_metric)
        hits         += in_top_k.sum().item()
        ndcg_sum     += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        n_users      += target_item.size(0)

        topk_items = scores.topk(k_recs, dim=1).indices.cpu().numpy()
        all_recs.append(topk_items)

    recs = np.concatenate(all_recs, axis=0)

    # Concept exposure: fraction of recommended items belonging to each concept's pool
    exposure = {c: float(np.isin(recs, np.array(list(pool))).mean())
                for c, pool in concept_pools.items()}

    return {
        f'hr@{k_metric}':   hits / n_users,
        f'ndcg@{k_metric}': ndcg_sum / n_users,
        'exposure':         exposure,
        'recs':             recs,
    }


GENRE_CONCEPT_IDX = {g: i for i, g in enumerate(genre_concepts)}


TARGET_GENRES = ['Comedy']     # try one genre first
BOOST    =  1.6              # genres tend to need a bigger boost than eras
                               # because individual genre activations are smaller
SUPPRESS = 1               # leave other genres alone

print("Evaluating baseline (no steering)...")
base = evaluate_concept_steered(
    cbm, test_data, genre_pools, GENRE_CONCEPT_IDX,
    target_concepts=TARGET_GENRES, boost=1.0, suppress=1.0,
    k_metric=20, k_recs=20
)

print(f"Evaluating steered toward {TARGET_GENRES} (boost×{BOOST})...")
steer = evaluate_concept_steered(
    cbm, test_data, genre_pools, GENRE_CONCEPT_IDX,
    target_concepts=TARGET_GENRES, boost=BOOST, suppress=SUPPRESS,
    k_metric=20, k_recs=20
)

# Print results
print("\n── ACCURACY ────────────────────────────────")
print(f"{'Metric':<14} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 48)
for key in ['hr@20', 'ndcg@20']:
    b, s = base[key], steer[key]
    print(f"{key:<14} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")

print("\n── GENRE EXPOSURE (top 10 by absolute change) ──")
print(f"{'Genre':<15} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 48)
deltas = sorted(genre_pools.keys(),
                key=lambda g: abs(steer['exposure'][g] - base['exposure'][g]),
                reverse=True)
for g in deltas[:10]:
    b, s = base['exposure'][g], steer['exposure'][g]
    marker = '  ←' if g in TARGET_GENRES else ''
    print(f"{g:<15} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}{marker}")

## recbole evalaution

In [79]:
import torch
import numpy as np
import copy
from recbole.evaluator import Collector, Evaluator

SCALE_FACTOR=1.2
POP_CONCEPT_IDX=18
# ── 1) CONFIGURE METRICS ──────────────────────────────────────────────────────
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity', 'TailPercentage', 'ShannonEntropy']
config['topk']         = [5, 10, 20]
config['valid_metric'] = 'NDCG@10'
config['eval_args']    = {'mode': 'full'}
config['tail_ratio']   = 0.2

collector = Collector(config)
evaluator = Evaluator(config)


# ── 2) STEERING-AWARE EVALUATION USING RECBOLE METRICS ────────────────────────
@torch.no_grad()
def evaluate_with_recbole(cbm, eval_data, concept_idx=None, scale=1.0):
    cbm.eval()
    collector.data_collect(eval_data)

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']
        positive_u   = torch.arange(item_seq.size(0), device=device)
        positive_i   = target_item

        # Forward through CBM with optional steering
        h, c_hat, _ = cbm(item_seq, item_seq_len)
        if concept_idx is not None:
            c_hat = c_hat.clone()
            c_hat[:, concept_idx] = c_hat[:, concept_idx] * scale
        h_hat  = cbm.reconstructor(c_hat)
        scores = cbm.score_items(h_hat)

        # Mask padding and history
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        # Hand batch off to RecBole's collector
        collector.eval_batch_collect(
            scores_tensor = scores,
            interaction   = interaction,
            positive_u    = positive_u,
            positive_i    = positive_i,
        )

    # ── Bypass the broken get_data_struct ─────────────────────────────────────
    # Move tensors to CPU; leave non-tensor entries (Counters, ints) alone.
    for key, val in list(collector.data_struct._data_dict.items()):
        if isinstance(val, torch.Tensor):
            collector.data_struct._data_dict[key] = val.cpu()
    print(f"collector.data_struct {collector.data_struct}")
    struct = copy.deepcopy(collector.data_struct)

    # Mirror the reset that get_data_struct normally performs
    for key in ["rec.topk", "rec.meanrank", "rec.score", "rec.items", "data.label"]:
        if key in collector.data_struct._data_dict:
            del collector.data_struct._data_dict[key]

    return evaluator.evaluate(struct)


# ── 3) RUN BASELINE AND STEERED EVALUATIONS ───────────────────────────────────
#print("Baseline (no steering):")
base = evaluate_with_recbole(cbm, test_data)
#for k, v in base.items():
    #print(f"  {k:<25} {v:.4f}")

#print(f"\nSteered (popularity × {SCALE_FACTOR}):")
steer = evaluate_with_recbole(cbm, test_data,
                              concept_idx=POP_CONCEPT_IDX,
                              scale=SCALE_FACTOR)
#for k, v in steer.items():
    #print(f"  {k:<25} {v:.4f}")


# ── 4) SIDE-BY-SIDE COMPARISON ────────────────────────────────────────────────
print("\n── COMPARISON ──────────────────────────────")
print(f"{'Metric':<25} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 59)
for key in base.keys():
    b, s = base[key], steer[key]
    print(f"{key:<25} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")

collector.data_struct 
Containing:
data.num_items
data.count_items
rec.items
rec.topk

collector.data_struct 
Containing:
data.num_items
data.count_items
rec.items
rec.topk


── COMPARISON ──────────────────────────────
Metric                      Baseline    Steered          Δ
───────────────────────────────────────────────────────────
recall@5                      0.1896     0.1816    -0.0080
recall@10                     0.2796     0.2755    -0.0041
recall@20                     0.3798     0.3760    -0.0038
ndcg@5                        0.1257     0.1196    -0.0061
ndcg@10                       0.1548     0.1500    -0.0048
ndcg@20                       0.1801     0.1754    -0.0047
mrr@5                         0.1048     0.0993    -0.0055
mrr@10                        0.1168     0.1119    -0.0049
mrr@20                        0.1237     0.1189    -0.0048
hit@5                         0.1896     0.1816    -0.0080
hit@10                        0.2796     0.2755    -0.0041
hit@20      

In [ ]:
from recbole.evaluator import Evaluator, Collector
# Check which metrics are causing the issue
print(config['metrics']) 

# Reset to standard metrics that RecBole recognizes
config['metrics'] = ['Recall', 'NDCG', 'MRR', 'Hit']
eval_collector = Collector(config)

In [ ]:
import time

# 1. Initialize variables
num_sample = 0
epoch_time = 0
device = next(model.parameters()).device 

for batch_idx, batched_data in enumerate(test_data):
    num_sample += len(batched_data)
    
    # Use time.time() to get the float value
    start_time = time.time()  
    
    # Unpack the batch (ensure this matches your dataloader format)
    interaction, _, positive_u, positive_i = batched_data
    
    # Get scores from your CBM or SASRec model
    scores = model.full_sort_predict(interaction.to(device))
    
    end_time = time.time()
    epoch_time += (end_time - start_time)
    
    # Collect batch results
    eval_collector.eval_batch_collect(
        scores, interaction, positive_u, positive_i
    )

# 2. Corrected call: only pass the model
eval_collector.model_collect(model) 

# 3. Finalize evaluation
struct = eval_collector.get_data_struct()
result = evaluator.evaluate(struct)

print(result)

In [ ]:
import time
import torch
from recbole.evaluator import Collector, Evaluator


# ── 1) CONFIGURE METRICS ──────────────────────────────────────────────────────
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit',
                          'ItemCoverage', 'GiniIndex',
                          'AveragePopularity', 'TailPercentage', 'ShannonEntropy']
config['topk']         = [5, 10, 20]
config['valid_metric'] = 'NDCG@10'
config['eval_args']    = {'mode': 'full'}
config['tail_ratio']   = 0.2


# ── 2) LOAD BEST CBM CHECKPOINT ───────────────────────────────────────────────
SAVE_PATH = 'best_cbm_ml-1m_SASREC.pt'

cbm = SASRecCBM(model, n_concepts=N_CONCEPTS, hidden_size=model.hidden_size).to(device)
checkpoint = torch.load(SAVE_PATH, map_location=device)
cbm.load_state_dict(checkpoint['model_state_dict'])
cbm.eval()

print(f"Loaded CBM from epoch {checkpoint['epoch']}")


# ── 3) JOINT EVALUATION ON IDENTICAL BATCHES ──────────────────────────────────
model.eval()
cbm.eval()
device = next(model.parameters()).device

# One collector per model, one shared evaluator
sasrec_collector = Collector(config)
cbm_collector    = Collector(config)
evaluator        = Evaluator(config)

# Tell each collector about the dataset
sasrec_collector.data_collect(test_data)
cbm_collector.data_collect(test_data)

num_sample  = 0
sasrec_time = 0.0
cbm_time    = 0.0

with torch.no_grad():
    for batch_idx, batched_data in enumerate(test_data):
        num_sample += len(batched_data)

        # Unpack the batch
        interaction, history_index, positive_u, positive_i = batched_data
        interaction = interaction.to(device)

        # ── SASRec scoring (uses full_sort_predict) ───────────────────────────
        t0 = time.time()
        sasrec_scores = model.full_sort_predict(interaction)

        # Reshape if flat
        if sasrec_scores.dim() == 1:
            sasrec_scores = sasrec_scores.view(interaction.length, -1)

        sasrec_time += time.time() - t0

        # ── CBM scoring (manual forward) ──────────────────────────────────────
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']

        t0 = time.time()
        h, c_hat, h_hat = cbm(item_seq, item_seq_len)
        cbm_scores      = cbm.score_items(h_hat)

        # Mask padding and history (full_sort_predict does this internally for SASRec,
        # but our CBM scorer does not — apply explicitly to keep both fair).
        cbm_scores[:, 0] = -float('inf')
        #cbm_scores.scatter_(1, item_seq, -float('inf'))

        cbm_time += time.time() - t0

        # ── Collect both batches ──────────────────────────────────────────────
        sasrec_collector.eval_batch_collect(
            sasrec_scores, interaction, positive_u, positive_i
        )
        cbm_collector.eval_batch_collect(
            cbm_scores, interaction, positive_u, positive_i
        )

# Finalize
sasrec_collector.model_collect(model)
cbm_collector.model_collect(cbm)

sasrec_struct = sasrec_collector.get_data_struct()
cbm_struct    = cbm_collector.get_data_struct()

sasrec_result = evaluator.evaluate(sasrec_struct)
cbm_result    = evaluator.evaluate(cbm_struct)


# ── 4) PRINT RESULTS ──────────────────────────────────────────────────────────
print(f"\nEvaluated {num_sample} samples")
print(f"SASRec time: {sasrec_time:.2f}s | CBM time: {cbm_time:.2f}s\n")

print("── SIDE-BY-SIDE COMPARISON ────────────────────────────────")
print(f"{'Metric':<25} {'SASRec':>12} {'CBM':>12} {'Δ (CBM - SASRec)':>20}")
print("─" * 73)

for key in sasrec_result.keys():
    s, c = sasrec_result[key], cbm_result[key]
    print(f"{key:<25} {s:>12.4f} {c:>12.4f} {c-s:>+20.4f}")

In [ ]:
batched_data

In [ ]:
from recbole.trainer import Trainer

# 1. Clean up the config for standard ranking
# We remove Gini/Diversity metrics to avoid the 'registration' error
config['metrics'] = ['Recall', 'NDCG', 'MRR', 'Hit']
config['topk'] = [5, 10, 20]

# 2. Ensure 'full' ranking (standard for SASRec research)
# This compares the target item against ALL other items
config['eval_args']['mode'] = 'full'

# 3. Initialize Trainer
trainer = Trainer(config, model)

# 4. Execute Evaluation
# load_best_model=False because you've already loaded the weights you want
test_result = trainer.evaluate(test_data, load_best_model=False, show_progress=True)

print("--- Final RecBole Metrics ---")
for metric, value in test_result.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
from recbole.trainer import Trainer
Trainer

In [ ]:
# 1. Initialize the trainer

# Pass override at load time — config gets these values BEFORE collector is built
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/sasrec_ml-1m.pth'
)

# Override metrics to only include standard ranking metrics
config['metrics'] = ['Recall', 'MRR', 'NDCG', 'Hit','ItemCoverage'] 
trainer = Trainer(config, model)
trainer.eval_collector.data_collect(train_data)   

test_result = trainer.evaluate(test_data, load_best_model=False)

print("Evaluation Results:")
print(test_result)

In [ ]:
trainer = Trainer(config, wrapped_base)
trainer.eval_collector = Collector(config)
trainer.evaluator      = Evaluator(config)

# Inspect
print("Evaluator metrics:", trainer.evaluator.metrics)
print("Collector register attributes:")
print("  has data.num_items:", trainer.eval_collector.data_struct._data_dict if hasattr(trainer.eval_collector, 'data_struct') else "n/a")
print("  register fields:", dir(trainer.eval_collector.register) if hasattr(trainer.eval_collector, 'register') else "n/a")

In [ ]:
dataset_name='ml-1mm'
item_path = f'./dataset/{dataset_name}/{dataset_name}.item'
items = pd.read_csv(item_path, sep='\t')

In [ ]:
items

In [ ]:
all_genres = set()
for row in items['genre:token_seq'].dropna():
    all_genres.update(str(row).split('|'))
genre_concepts = sorted(all_genres)

In [ ]:
genre_concepts

In [ ]:
# split each genre string by space, flatten, and deduplicate
all_unique_genres = sorted(set(
    g.strip()
    for genres in items['genre:token_seq'].dropna()
    for g in str(genres).split(' ')
    if g.strip()
))

In [ ]:
all_unique_genres

In [1]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
import logging

from recbole.model.sequential_recommender.sasrec_cbm import SASRec_CBM

# ── Config ───────────────────────────────────────────────────────────────────
config = Config(
    model='SASRec_CBM',
    dataset='ml-1mm',
    config_file_list=['CBM_config.yaml'],
    config_dict={                # override anything you want without editing YAML
        'lambda_concept': 1,
        'lambda_recon':   4,
      
    },
)

init_seed(config['seed'], config['reproducibility'])
init_logger(config)
logger = logging.getLogger()

# ── Data ─────────────────────────────────────────────────────────────────────
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

# ── Model ────────────────────────────────────────────────────────────────────
model = SASRec_CBM(config, train_data.dataset).to(config['device'])

# ── Trainer ──────────────────────────────────────────────────────────────────
trainer = Trainer(config, model)
trainer.eval_collector.data_collect(train_data)

# ── Train ────────────────────────────────────────────────────────────────────
best_valid_score, best_valid_result = trainer.fit(
    train_data, valid_data,
    saved=True,                  # save checkpoint
    show_progress=True,
)
print(f"\nBest valid: {best_valid_score:.4f}")
print(f"  metrics:  {best_valid_result}")

# ── Test ─────────────────────────────────────────────────────────────────────
test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=True)
print(f"\nTest result: {test_result}")

/home/mvarasteh/post-hoc/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/home/mvarasteh/post-hoc/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 

[CBM] Loaded 3417 items × 25 concepts


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mvarasteh/.netrc.
wandb: Currently logged in as: m-varasteh92 (m-varasteh92-university-of-colorado-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train     0:   0%|                                                          | 0/480 [00:00<?, ?it/s]/home/mvarasteh/post-hoc/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


out_shape: tensor([[0.1600, 0.1200, 0.0200,  ..., 0.1600, 0.8200, 0.0000],
        [0.1400, 0.1200, 0.0400,  ..., 0.1429, 0.8163, 0.0000],
        [0.1200, 0.0400, 0.0600,  ..., 0.0612, 0.9184, 0.0000],
        ...,
        [0.1800, 0.0800, 0.0200,  ..., 0.0800, 0.8800, 0.0000],
        [0.1200, 0.0400, 0.0000,  ..., 0.0208, 0.8125, 0.0208],
        [0.1250, 0.1250, 0.0000,  ..., 0.0833, 0.8333, 0.0000]],
       device='cuda:0')


...

out_shape: tensor([[0.2200, 0.1200, 0.0400,  ..., 0.0612, 0.8776, 0.0204],
        [0.0400, 0.0000, 0.0400,  ..., 0.0204, 0.9388, 0.0000],
        [0.2200, 0.0800, 0.0200,  ..., 0.0851, 0.8298, 0.0000],
        ...,
        [0.1200, 0.0800, 0.0200,  ..., 0.0400, 0.9600, 0.0000],
        [0.2600, 0.0800, 0.0400,  ..., 0.1400, 0.8400, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.1429, 0.8571, 0.0000]],
       device='cuda:0')


Train     0:  39%|████████▉              | 187/480 [00:07<00:11, 24.51it/s, GPU RAM: 1.13 G/15.72 G]:  39%|█████████              | 189/480 [00:07<00:12, 23.97it/s, GPU RAM: 1.13 G/15.72 G]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x76a36e5a72b0>> (for post_run_cell), with arguments args (<ExecutionResult object at 76a4f4509300, execution_count=1 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 76a4f4509000, raw_cell="from recbole.config import Config
from recbole.dat.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bganon/home/mvarasteh/post-hoc/Concept_predictor.ipynb#X65sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: [Errno 104] Connection reset by peer

In [ ]:
trainer

In [ ]:
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.model.sequential_recommender.sasrec_cbm import SASRec_CBM

# ── 1. Rebuild config (same as how you originally trained) ───────────────────
config = Config(
    model='SASRec_CBM',
    dataset='ml-1mm',
    config_file_list=['CBM_config.yaml'],
    config_dict={
        'train_neg_sample_args': None,
        'initializer_range': 0.02,
        'layer_norm_eps': 1.0e-12,
        # add any other overrides you used during training
    },
)

# ── 2. Build data ────────────────────────────────────────────────────────────
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

# ── 3. Build model (this loads pretrained SASRec weights via base_path) ──────
model = SASRec_CBM(config, dataset).to(config['device'])

# ── 4. Load your saved CBM weights ───────────────────────────────────────────
ckpt = torch.load(
    "/home/mvarasteh/post-hoc/best_cbm_ml-1m_SASREC_3pop.pt",
    map_location=config['device'],
    weights_only=False,
)
print(f"[load] epoch:   {ckpt['epoch']}")
print(f"[load] metrics: {ckpt['metrics']}")
print(f"[load] hidden_size: {ckpt['hidden_size']}, n_concepts: {ckpt['n_concepts']}")

# Note the key is 'model_state_dict', not 'state_dict'
model.load_state_dict(ckpt['model_state_dict'])


# ── 5. Evaluate ──────────────────────────────────────────────────────────────
trainer = Trainer(config, model)
trainer.eval_collector.data_collect(train_data)

test_result = trainer.evaluate(
    test_data,
    load_best_model=False,
    show_progress=False,
)

# Pretty-print
print(f"\n{'Metric':<25} {'Value':>10}")
print("-" * 38)
for k, v in test_result.items():
    print(f"{k:<25} {v:>10.4f}")

In [1]:
from recbole.quick_start import load_data_and_model
from recbole.trainer import Trainer
from recbole.evaluator import Collector, Evaluator

# Load model + config + data (all derived from the saved checkpoint)
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/sasrec_ml-1m.pth'
)

# Override only the metrics you want to change
config['metrics']      = ['Recall', 'NDCG', 'MRR', 'Hit', 'ItemCoverage', 'GiniIndex']
config['topk']         = [ 10]
config['valid_metric'] = 'NDCG@10'

# Build trainer
trainer = Trainer(config, model)

# Trainer caches eval_collector and evaluator in __init__, so rebuild them
# after mutating config['metrics']
trainer.eval_collector = Collector(config)
trainer.evaluator      = Evaluator(config)
trainer.eval_collector.data_collect(train_data)

# Evaluate
test_result = trainer.evaluate(test_data, load_best_model=False)

print("Evaluation Results:")
for k, v in test_result.items():
    print(f"  {k:<25} {v:.4f}")

/home/mvarasteh/.conda/envs/popsteer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-12 17:42:40,965	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-12 17:42:41,200	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
12 May 17:42    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = dataset/ml-1mm
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = ad

Evaluation Results:
  recall@10                 0.2483
  ndcg@10                   0.1345
  mrr@10                    0.0998
  hit@10                    0.2483
  itemcoverage@10           0.7185
  giniindex@10              0.7288


In [ ]:
/home/mvarasteh/post-hoc/saved/SASRec_CBM-May-12-2026_18-21-31.pth

In [3]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, SASRec_model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/sasrec_ml-1m.pth'
)
model.eval()
device = config['device']

12 May 18:32    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = dataset/ml-1mm
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'NDCG', 'Hit', 'Deep_LT_Coverage', 'GiniIndex', 'AveragePopularity', 'ItemCoverage', 'NDCGTail', 'NDCGHead', 'NDCGMid']
topk = [10]
valid_metric = NDCG@10
valid_metric_

In [2]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, CBM_model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/SASRec_CBM-May-12-2026_18-21-31.pth'
)
model.eval()
device = config['device']

12 May 18:31    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = ./dataset/ml-1mm
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = True

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 300
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'NDCG', 'MRR', 'Hit', 'ItemCoverage', 'GiniIndex', 'AveragePopularity', 'ConceptAccuracySoft', 'ConceptAccuracyStrict']
topk = [5, 10, 20]
valid_metric = NDCG@10
vali

[build] Building concept lookups...
  pop_set:   342 items (≥ 751 interactions)
  mid_set:   2725 items
  niche_set: 349 items (1–17 interactions)
  Items in metadata file: 3883, matched to internal IDs: 3416
  N_CONCEPTS = 25  (18 genres + 7 scalars)
  genres: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
[build] Computing per-item concept vectors for 3417 items...

[build] Saved per-item concept matrix ((3417, 25)) → ./dataset/ml-1mm/saved_concept_individual_items.pkl

[build] Diagnostics — items with NO ...
    ... genre features:  1 (should be ~1, padding)
    ... era features:    1 (only items missing year)
    ... tier features:   1 (only items w/ 0 interactions)
    mean genres/item:    1.71

[build] Spot check — item 1's concept vector:
    Drama                1.0000
    popularity           1.0000
    retro                1

In [37]:
(SASRec_model.trm_encoder.layer[0].multi_head_attention.value.weight-CBM_model.trm_encoder.layer[0].multi_head_attention.value.weight).sum()

tensor(0., device='cuda:0', grad_fn=<SumBackward0>)

In [34]:
CBM_model.trm_encoder.layer[0].multi_head_attention.query.weight

Parameter containing:
tensor([[-0.0441,  0.0010, -0.0357,  ...,  0.0284, -0.0158, -0.0531],
        [-0.0138, -0.1460,  0.1986,  ..., -0.0980,  0.0300,  0.0527],
        [-0.1099, -0.0334, -0.1148,  ...,  0.0337, -0.1050,  0.0185],
        ...,
        [-0.0867, -0.0297, -0.4245,  ...,  0.0027,  0.0469,  0.0245],
        [-0.0161,  0.0723,  0.2851,  ..., -0.0985, -0.1300,  0.0622],
        [-0.0497, -0.1140,  0.1699,  ..., -0.0669, -0.0286,  0.0826]],
       device='cuda:0')

In [40]:
CBM_model

SASRec_CBM(
  (item_embedding): Embedding(3417, 128, padding_idx=0)
  (position_embedding): Embedding(50, 128)
  (trm_encoder): TransformerEncoder(
    (layer): ModuleList(
      (0-1): 2 x TransformerLayer(
        (multi_head_attention): MultiHeadAttention(
          (query): Linear(in_features=128, out_features=128, bias=True)
          (key): Linear(in_features=128, out_features=128, bias=True)
          (value): Linear(in_features=128, out_features=128, bias=True)
          (softmax): Softmax(dim=-1)
          (attn_dropout): Dropout(p=0, inplace=False)
          (dense): Linear(in_features=128, out_features=128, bias=True)
          (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
          (out_dropout): Dropout(p=0, inplace=False)
        )
        (feed_forward): FeedForward(
          (dense_1): Linear(in_features=128, out_features=256, bias=True)
          (dense_2): Linear(in_features=256, out_features=128, bias=True)
          (LayerNorm): LayerNorm((128,